# Weather Features -- Open-Meteo (52-Week Lag)

Generates `features/ds_weather.parquet` with historical climate variables per macro-region.

## Design: why a 52-week lag?

| Scenario | Problem |
|---|---|
| Real future climate | Unavailable - Open-Meteo forecast covers only **16 days** (< 4-week horizon) |
| Current climate directly | Creates leakage: test and prediction sets would not have this data in production |
| **Climate lagged 52 weeks** | Always available: uses the **same period from the prior year** as a seasonal proxy |

**Flow:**
```
Dataset range   : Oct/2022 -> Oct/2024
API fetch       : Oct/2021 -> Oct/2023  (52 weeks earlier)
After +52w shift: Oct/2022 -> Oct/2024  (aligned with series)
Future forecast : Nov/2024 uses Nov/2023 climate  (historical data available)
```

**Limitation:** does not capture inter-annual anomalies (El Nino, exceptional heatwaves).
For pharma demand, the seasonal pattern dominates -- this trade-off is acceptable.

In [ ]:
import os
import time
import warnings
import requests
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../src')
from config import LoadConfig
cfg = LoadConfig()

In [ ]:
# -- Config ----------------------------------------------------------------
PATH_WEATHER  = '../../data/gold/forecasting/features/ds_weather.parquet'
os.makedirs(os.path.dirname(PATH_WEATHER), exist_ok=True)

LAG_WEEKS     = 52   # temporal shift applied after fetching
HORIZON_WEEKS = 4    # extra weeks to ensure future coverage
DEMAND_TYPES  = ['smooth', 'erratic', 'intermittent', 'lumpy']

# Representative coordinates per macro-region (capital or most populous city)
REGION_COORDS = {
    1: {'name': 'Centro-Oeste', 'lat': -15.78, 'lon': -47.93},  # Brasilia/DF
    2: {'name': 'Nordeste',     'lat':  -8.05, 'lon': -34.90},  # Recife/PE
    3: {'name': 'Norte',        'lat':  -3.10, 'lon': -60.02},  # Manaus/AM
    4: {'name': 'Sudeste',      'lat': -23.55, 'lon': -46.63},  # Sao Paulo/SP
    5: {'name': 'Sul',          'lat': -25.43, 'lon': -49.27},  # Curitiba/PR
}

DAILY_VARS = [
    'temperature_2m_max',
    'temperature_2m_min',
    'temperature_2m_mean',
    'precipitation_sum',
    'windspeed_10m_max',
]

## 1. Determine Date Range

In [ ]:
# Read the actual week dates from the refined series
all_weeks = pd.concat([
    cfg.load_forecast('datasets', 'refined', dt)[['unique_id', 'ds']]
    for dt in DEMAND_TYPES
])['ds'].drop_duplicates().sort_values()

ds_min = all_weeks.min()
ds_max = all_weeks.max()

# Fetch range = [ds_min - LAG_WEEKS, ds_max + HORIZON_WEEKS - LAG_WEEKS]
# After the +LAG_WEEKS shift this covers [ds_min, ds_max + HORIZON_WEEKS]
fetch_start = ds_min - pd.Timedelta(weeks=LAG_WEEKS)
fetch_end   = ds_max + pd.Timedelta(weeks=HORIZON_WEEKS) - pd.Timedelta(weeks=LAG_WEEKS)

print(f'Dataset        : {ds_min.date()}  ->  {ds_max.date()}')
print(f'API fetch      : {fetch_start.date()}  ->  {fetch_end.date()}')
print(f'After +{LAG_WEEKS}w shift : '
      f'{(fetch_start + pd.Timedelta(weeks=LAG_WEEKS)).date()}  ->  '
      f'{(fetch_end + pd.Timedelta(weeks=LAG_WEEKS)).date()}')
print(f'Unique weeks   : {len(all_weeks):,}')

## 2. Fetch Open-Meteo Historical Archive

In [ ]:
ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'


def fetch_daily_weather(lat, lon, start, end, variables, retries=3):
    # Free service (no API key) with a soft rate limit
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': start,
        'end_date':   end,
        'daily':      ','.join(variables),
        'timezone':   'America/Sao_Paulo',
    }
    for attempt in range(retries):
        try:
            r = requests.get(ARCHIVE_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()['daily']
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['time'])
            return df.drop(columns='time').set_index('date')
        except requests.HTTPError:
            if r.status_code == 429:
                time.sleep(2 ** attempt)
            else:
                raise
    raise RuntimeError(f'Failed after {retries} retries')


def aggregate_to_weekly(df_daily: 'pd.DataFrame') -> 'pd.DataFrame':
    df = df_daily.resample('W-MON').agg({
        'temperature_2m_max':  'max',
        'temperature_2m_min':  'min',
        'temperature_2m_mean': 'mean',
        'precipitation_sum':   'sum',
        'windspeed_10m_max':   'max',
    }).reset_index().rename(columns={
        'date':                'ds',
        'temperature_2m_max':  'temp_max',
        'temperature_2m_min':  'temp_min',
        'temperature_2m_mean': 'temp_mean',
        'precipitation_sum':   'precip',
        'windspeed_10m_max':   'wind_max',
    })
    df['is_cold']  = (df['temp_mean'] < 15).astype(int)
    df['is_hot']   = (df['temp_mean'] > 30).astype(int)
    df['is_rainy'] = (df['precip'] > 50).astype(int)
    return df

In [ ]:
start_str = fetch_start.strftime('%Y-%m-%d')
end_str   = fetch_end.strftime('%Y-%m-%d')

raw_by_region = {}

for region_id, info in REGION_COORDS.items():
    print(f'Fetching region {region_id} - {info["name"]} ({info["lat"]}, {info["lon"]})...', end=' ')
    df_daily  = fetch_daily_weather(info['lat'], info['lon'], start_str, end_str, DAILY_VARS)
    df_weekly = aggregate_to_weekly(df_daily)
    df_weekly['region_id'] = region_id
    raw_by_region[region_id] = df_weekly
    print(f'{len(df_weekly)} weeks OK')
    time.sleep(0.5)  # respect soft rate limit

df_raw = pd.concat(raw_by_region.values(), ignore_index=True)
print(f'\nTotal: {len(df_raw):,} rows  |  columns: {df_raw.columns.tolist()}')

## 3. Apply 52-Week Lag Shift

```
weather_lag52[t] = actual climate at (t - 52 weeks)
```

Advancing the raw dates forward by 52 weeks aligns each row with the
corresponding period in the model's training and prediction range.

In [ ]:
WEATHER_COLS = ['temp_max', 'temp_min', 'temp_mean', 'precip', 'wind_max',
                'is_cold', 'is_hot', 'is_rainy']

df_weather = df_raw.copy()

# Advance dates by 52 weeks -> align with the model period
df_weather['ds'] = df_weather['ds'] + pd.Timedelta(weeks=LAG_WEEKS)

# Keep only weeks within [ds_min, ds_max + horizon]
valid_range_end = ds_max + pd.Timedelta(weeks=HORIZON_WEEKS)
df_weather = df_weather[
    (df_weather['ds'] >= ds_min) &
    (df_weather['ds'] <= valid_range_end)
].reset_index(drop=True)

print(f'After shift: {len(df_weather):,} rows')
print(f'Date range : {df_weather["ds"].min().date()}  ->  {df_weather["ds"].max().date()}')

## 4. Validation

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 7))

plot_vars = [
    ('temp_mean', 'Mean Temperature (C)', 'tomato'),
    ('precip',    'Precipitation (mm)',    'steelblue'),
    ('wind_max',  'Max Wind Speed (km/h)', 'seagreen'),
]

for col_idx, (var, label, colour) in enumerate(plot_vars):
    for row_idx, region_id in enumerate([4, 2]):  # Sudeste and Nordeste
        ax  = axes[row_idx][col_idx]
        sub = df_weather[df_weather['region_id'] == region_id].sort_values('ds')
        ax.plot(sub['ds'], sub[var], color=colour, linewidth=0.8)
        region_name = REGION_COORDS[region_id]['name']
        ax.set_title(f'{label} - {region_name}', fontsize=9)
        ax.tick_params(labelsize=8)

fig.suptitle('Weather Features -- 52-week lag applied', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Save

In [ ]:
df_weather.to_parquet(PATH_WEATHER, index=False)

print(f'Saved  : {PATH_WEATHER}')
print(f'Shape  : {df_weather.shape}')
print(f'Columns: {df_weather.columns.tolist()}')
print(f'Regions: {sorted(df_weather["region_id"].unique())}')
print(f'Weeks  : {df_weather["ds"].nunique():,}')